# 6차시. 그룹별 집계와 데이터 분석

## 1. 6차시 학습목표
groupby 그룹별 집계, agg 여러 통계, transform 그룹 기준 파생 변수까지 이번 차시 목표를 확인합니다.

## 2. 선수지식 — 5차시의 그룹별 계산
tips와 5차시 결과를 불러오고, 5차시 방식대로 어린이 생존률을 한 그룹만 계산합니다.

In [1]:
import pandas as pd                          # 판다스 불러오기
import numpy as np                           # np.where 사용

tips = pd.read_csv("data/raw/tips.csv")      # tips 데이터 불러오기
feat = pd.read_csv("data/processed/titanic_features.csv")   # 5차시 결과

In [2]:
is_kid = feat["age_group"] == "어린이"                   # 그룹 하나씩 조건 만들기
child_rate = feat.loc[is_kid, "survived"].mean()        # 그 그룹의 생존률
print("어린이 생존률:", round(child_rate, 2))

어린이 생존률: 0.59


## 3. groupby — 나누고 · 계산하고 · 합친다
요일별로 묶은 결과의 타입과 그룹별 행 수, 그룹 개수를 확인합니다.

In [ ]:
g = tips.groupby("day")
# g.size()
g.ngroups

4

## 4. 그룹별 집계 — 한 열 · 한 함수
요일별 팁 합계, 성별 평균 결제 금액, 시간대별 최대 인원을 구합니다.

In [17]:
tips.groupby("day")["tip"].sum()

day
Fri      51.96
Sat     260.40
Sun     247.39
Thur    171.83
Name: tip, dtype: float64

In [18]:
tips.groupby("sex")["total_bill"].mean()

sex
Female    18.056897
Male      20.744076
Name: total_bill, dtype: float64

In [19]:
tips.groupby("time")["size"].max()

time
Dinner    6
Lunch     6
Name: size, dtype: int64

## 5. 여러 기준 — MultiIndex
요일과 시간대 두 기준으로 묶어 평균 팁을 구합니다.

In [55]:
x = tips.groupby( ["day" , "time"] )["tip"].mean()
x

day   time  
Fri   Dinner    2.940000
      Lunch     2.382857
Sat   Dinner    2.993103
Sun   Dinner    3.255132
Thur  Dinner    3.000000
      Lunch     2.767705
Name: tip, dtype: float64

## 6. 그룹을 열로 — as_index=False
같은 집계를 그룹 기준이 일반 열로 들어간 평평한 표로 받습니다.

In [63]:
tips.groupby(["day" , "time"] , as_index=False)["tip"].mean()

,day,time,tip
0,Fri,Dinner,2.940000
1,Fri,Lunch,2.382857
2,Sat,Dinner,2.993103
3,Sun,Dinner,3.255132
4,Thur,Dinner,3.000000
5,Thur,Lunch,2.767705


## 7. 여러 열 한 번에 집계
요일별 결제 금액과 팁의 평균을 한 번에 구합니다.

In [68]:
tips.groupby("day")[["total_bill" , "tip"]].mean()

,total_bill,tip
day,,
Fri,17.151579,2.734737
Sat,20.441379,2.993103
Sun,21.410000,3.255132
Thur,17.682742,2.771452


## 8. agg — 여러 함수 한 번에
요일별 팁의 평균·최대·개수를 한 번에 구합니다.

In [74]:
tips.groupby("day")["tip"].agg( ["mean" , "max"] )

,mean,max
day,,
Fri,2.734737,4.73
Sat,2.993103,10.00
Sun,3.255132,6.50
Thur,2.771452,6.70


## 9. agg — 열마다 다른 함수
결제 금액은 합계, 팁은 평균·최대, 인원은 평균으로 열마다 다른 통계를 구합니다.

In [77]:
tips.groupby("day").agg({
    "total_bill" : "sum" ,
    "tip" : ["mean" , "max"] ,
})

total_bill       tip       
            sum      mean    max
day                             
Fri      325.88  2.734737   4.73
Sat     1778.40  2.993103  10.00
Sun     1627.16  3.255132   6.50
Thur    1096.33  2.771452   6.70

## 10. agg — 결과 열에 이름 붙이기
named aggregation으로 한글 열 이름이 붙은 요일별 요약표를 만듭니다.

In [79]:
tips.groupby("day").agg(
    방문수=("tip" , "count") ,
    평균팁=("tip" , "mean") ,
    총매출=("total_bill" , "sum") ,
)

,방문수,평균팁,총매출
day,,,
Fri,19,2.734737,325.88
Sat,87,2.993103,1778.40
Sun,76,3.255132,1627.16
Thur,62,2.771452,1096.33


## 11. agg — 직접 만든 함수
최댓값과 최솟값의 차이를 구하는 함수를 만들어 요일별 팁 범위를 구합니다.

In [81]:
def value_range(sr):
    return sr.max() - sr.min()

tips.groupby("day").agg(
    팁_최소=("tip" , "min") ,
    팁_최대=("tip" , "max") ,
    팁_범위=("tip" , value_range) ,
)

,팁_최소,팁_최대,팁_범위
day,,,
Fri,1.00,4.73,3.73
Sat,1.00,10.00,9.00
Sun,1.01,6.50,5.49
Thur,1.25,6.70,5.45


## 12. transform — 그룹 결과를 행마다 돌려주기
agg와 transform 결과의 행 수를 비교하고, 요일 평균 팁을 새 열로 붙입니다.

In [102]:
a = tips.groupby("day")["tip"].mean()
b = tips.groupby("day")["tip"].transform("mean")

tips["day_tip_avg"] = b
tips

,total_bill,tip,sex,smoker,day,time,size,day_tip_avg
0,16.99,1.01,Female,No,Sun,Dinner,2,3.255132
1,10.34,1.66,Male,No,Sun,Dinner,3,3.255132
2,21.01,3.50,Male,No,Sun,Dinner,3,3.255132
3,23.68,3.31,Male,No,Sun,Dinner,2,3.255132
4,24.59,3.61,Female,No,Sun,Dinner,4,3.255132
...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,2.993103
240,27.18,2.00,Female,Yes,Sat,Dinner,2,2.993103
241,22.67,2.00,Male,Yes,Sat,Dinner,2,2.993103
242,17.82,1.75,Male,No,Sat,Dinner,2,2.993103


## 13. transform 활용 — 그룹 평균 대비
각 손님의 팁을 요일 평균과 비교해 평균 이상·미만을 표시합니다.

In [105]:
tips["tip_minus_avg"] = tips["tip"] - tips["day_tip_avg"]
tips["is_good"] = np.where( tips["tip_minus_avg"] >= 0 , 'GOOD' , 'BAD' )
tips

,total_bill,tip,sex,smoker,day,time,size,day_tip_avg,tip_minus_avg,is_good
0,16.99,1.01,Female,No,Sun,Dinner,2,3.255132,-2.245132,BAD
1,10.34,1.66,Male,No,Sun,Dinner,3,3.255132,-1.595132,BAD
2,21.01,3.50,Male,No,Sun,Dinner,3,3.255132,0.244868,GOOD
3,23.68,3.31,Male,No,Sun,Dinner,2,3.255132,0.054868,GOOD
4,24.59,3.61,Female,No,Sun,Dinner,4,3.255132,0.354868,GOOD
...,...,...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,2.993103,2.926897,GOOD
240,27.18,2.00,Female,Yes,Sat,Dinner,2,2.993103,-0.993103,BAD
241,22.67,2.00,Male,Yes,Sat,Dinner,2,2.993103,-0.993103,BAD
242,17.82,1.75,Male,No,Sat,Dinner,2,2.993103,-1.243103,BAD


## 14. transform 활용 — 그룹 내 비중
테이블별 결제 금액이 그 요일 매출에서 차지하는 비중을 구하고 합이 100인지 확인합니다.

In [106]:
# 이거 완료하고 집에가기

## 15. transform 활용 — 그룹별 중앙값으로 결측치 채우기
객실 등급·성별 그룹의 중앙값으로 나이 결측치를 채웁니다.

In [107]:
# 이거 완료하고 집에가기

## 16. 종합 실습 — 그룹별 생존 리포트
연령대별 생존률과 객실 등급·성별 생존 요약표를 만듭니다.

In [108]:
# 연령대별 생존률

In [109]:
# 객실등급 , 성별 생존률

## 17. 종합 실습 — 등급 평균 대비 요금 + 리포트 저장
등급 평균 대비 요금을 계산하고, 요약표를 outputs 폴더에 저장합니다.

## 18. 6차시 정리
groupby·agg·transform을 정리합니다.